# Challenge 2 — Kernel cuántico con `pytket` desde cero

Este notebook crea una ejecución **completamente nueva y reproducible** a partir de `water_potability.csv`.

Principios metodológicos:

1. El único archivo de entrada es el dataset original.
2. Se crea un holdout final bloqueado mediante SHA-256.
3. La selección de variables se realiza en un pool independiente.
4. Las 64 muestras cuánticas se extraen de otro pool, sin datos sintéticos.
5. Se usan **4 variables = 4 qubits**.
6. El circuito se implementa únicamente con `pytket`; no se importa Qiskit.
7. La validación cruzada ajusta imputación y escalado dentro de cada fold.
8. Todos los splits, hashes, kernels, métricas y circuitos se guardan como artefactos nuevos.

El kernel utilizado es

\[
K_{ij}=\left|\langle\phi(x_i)\mid\phi(x_j)\rangle\right|^2,
\]

y se entrega a `SVC(kernel="precomputed")`.

In [22]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
# Instalación reproducible. pytket 2.x mantiene las APIs empleadas abajo.
%pip install -q "pytket>=2.18,<3" "scikit-learn>=1.5,<2" pandas numpy matplotlib

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

from pytket import Circuit
from pytket.circuit import OpType
from pytket.circuit.display import render_circuit_as_html, render_circuit_jupyter
from pytket.qasm import circuit_to_qasm_str

print("Python:", sys.version)
try:
    import pytket
    print("pytket:", pytket.__version__)
except Exception:
    print("pytket importado correctamente")

assert not any(name.startswith("qiskit") for name in sys.modules), "Este notebook no debe importar Qiskit."

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
pytket: 2.18.1


## 1. Configuración

Las semillas tienen funciones separadas para que un cambio en un paso no altere silenciosamente los demás.
El directorio de artefactos es exclusivo de esta ejecución y se reinicia al comenzar.

In [ ]:
SEED_SPLIT = 20260722
SEED_SELECTOR_SPLIT = 20260723
SEED_FEATURE_MODEL = 20260724
SEED_QUANTUM_SAMPLE = 20260725
SEED_FOLDS = 20260726
SEED_SHOTS = 20260727

TARGET = "Potability"
N_QUBITS = 5
QUANTUM_ROWS = 64
N_SPLITS = 4
RESET_ARTIFACTS = True

# Solo se revisan rutas exactas del dataset; no se buscan artefactos anteriores.
DATASET_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/water_potability.csv"),
]

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATASET_PATH = next((p for p in DATASET_CANDIDATES if p.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        "No se encontró water_potability.csv. Súbalo a /content o colóquelo en "
        "/content/drive/MyDrive/Colab Notebooks/."
    )

ARTIFACT_DIR = DATASET_PATH.parent / "artifacts_pytket_from_scratch_v1"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
if RESET_ARTIFACTS:
    # Limpiar solo el contenido del directorio dedicado; conservar la carpeta de Drive.
    for child in ARTIFACT_DIR.iterdir():
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()

print("Único input:", DATASET_PATH)
print("Artefactos nuevos:", ARTIFACT_DIR)

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def write_json(path: Path, obj: dict) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


def save_csv(df: pd.DataFrame, name: str) -> Path:
    path = ARTIFACT_DIR / name
    df.to_csv(path, index=False)
    return path


def file_record(path: Path) -> dict:
    return {"file": path.name, "sha256": sha256_file(path), "bytes": path.stat().st_size}


def verify_records(records: list[dict]) -> None:
    for rec in records:
        path = ARTIFACT_DIR / rec["file"]
        assert path.exists(), f"Falta {path}"
        assert sha256_file(path) == rec["sha256"], f"Hash inválido: {path.name}"


def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

## 2. Leer y congelar el dataset original

Se guarda una copia exacta dentro del nuevo directorio de artefactos. El hash del archivo original y el de la copia deben coincidir.

In [ ]:
raw_bytes_hash = sha256_file(DATASET_PATH)
SNAPSHOT_PATH = ARTIFACT_DIR / "water_potability_input_snapshot.csv"
shutil.copy2(DATASET_PATH, SNAPSHOT_PATH)
assert sha256_file(SNAPSHOT_PATH) == raw_bytes_hash

df = pd.read_csv(SNAPSHOT_PATH).reset_index(names="source_index")
EXPECTED_COLUMNS = [
    "ph", "Hardness", "Solids", "Chloramines", "Sulfate",
    "Conductivity", "Organic_carbon", "Trihalomethanes",
    "Turbidity", "Potability",
]
assert [c for c in df.columns if c != "source_index"] == EXPECTED_COLUMNS
assert len(df) == 3276
assert set(df[TARGET].unique()) == {0, 1}

FEATURES_ALL = [c for c in EXPECTED_COLUMNS if c != TARGET]
print(df.shape)
display(df.head())
display(df[TARGET].value_counts().rename("count").to_frame())
display(df[FEATURES_ALL].isna().sum().rename("missing").to_frame())

## 3. Nuevos splits independientes

Se crean tres grupos disjuntos:

- **Holdout bloqueado (20%)**: reservado para una evaluación posterior.
- **Pool selector (60% del total)**: se usa únicamente para elegir las cuatro variables.
- **Pool candidato cuántico (20% del total)**: de aquí se extraen las 64 observaciones.

La selección de variables nunca ve las etiquetas de las 64 filas cuánticas.

In [ ]:
all_idx = np.arange(len(df))
y_all = df[TARGET].to_numpy(dtype=int)

dev_idx, holdout_idx = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=SEED_SPLIT,
    stratify=y_all,
)
selector_idx, quantum_candidate_idx = train_test_split(
    dev_idx,
    test_size=0.25,
    random_state=SEED_SELECTOR_SPLIT,
    stratify=y_all[dev_idx],
)

sets = [set(selector_idx), set(quantum_candidate_idx), set(holdout_idx)]
assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2])
assert sum(map(len, sets)) == len(df)

roles = pd.DataFrame({"source_index": df["source_index"], "role": ""})
roles.loc[selector_idx, "role"] = "selector_pool"
roles.loc[quantum_candidate_idx, "role"] = "quantum_candidate_pool"
roles.loc[holdout_idx, "role"] = "locked_holdout"
assert (roles["role"] != "").all()

paths_initial = []
paths_initial.append(save_csv(df.iloc[selector_idx].sort_values("source_index"), "selector_pool_raw.csv"))
paths_initial.append(save_csv(df.iloc[quantum_candidate_idx].sort_values("source_index"), "quantum_candidate_pool_raw.csv"))
paths_initial.append(save_csv(df.iloc[holdout_idx].sort_values("source_index"), "locked_holdout_raw.csv"))
paths_initial.append(save_csv(roles.sort_values("source_index"), "split_assignments.csv"))

split_summary = (
    roles.merge(df[["source_index", TARGET]], on="source_index")
         .groupby(["role", TARGET]).size().rename("count").reset_index()
)
display(split_summary)

## 4. Selección independiente de cuatro variables

Se imputan medianas en el **pool selector** y se entrena un `ExtraTreesClassifier` determinista. Sus importancias se utilizan únicamente para ordenar las nueve variables y escoger las primeras cuatro.

Esta reducción baja el estado de \(2^9=512\) amplitudes a \(2^4=16\), haciendo el estudio del kernel mucho más ligero.

In [ ]:
selector_df = df.iloc[selector_idx].copy()
selector_imputer = SimpleImputer(strategy="median")
X_selector_imp = selector_imputer.fit_transform(selector_df[FEATURES_ALL])
y_selector = selector_df[TARGET].to_numpy(dtype=int)

selector_model = ExtraTreesClassifier(
    n_estimators=500,
    random_state=SEED_FEATURE_MODEL,
    class_weight="balanced",
    n_jobs=-1,
)
selector_model.fit(X_selector_imp, y_selector)

importance_df = pd.DataFrame({
    "feature": FEATURES_ALL,
    "importance": selector_model.feature_importances_,
    "imputation_median": selector_imputer.statistics_,
}).sort_values(["importance", "feature"], ascending=[False, True], ignore_index=True)

# --- Custom logic to include 'Conductivity' and select N_QUBITS features ---
forced_feature = 'Conductivity'

# Start with the forced feature
SELECTED_FEATURES = [forced_feature]

# Get other features by importance, excluding the forced feature
other_features_df = importance_df[importance_df['feature'] != forced_feature]

# Select the remaining N_QUBITS - 1 features from the top of the importance list
num_additional_features = N_QUBITS - len(SELECTED_FEATURES)
if num_additional_features > 0:
    SELECTED_FEATURES.extend(other_features_df.head(num_additional_features)["feature"].tolist())

# Ensure we have exactly N_QUBITS unique features. This step handles potential overlaps
# if 'Conductivity' was already in the top (N_QUBITS-1) features by importance.
# However, given 'Conductivity's rank, it won't be in the top N_QUBITS-1.
# If the list ends up with more than N_QUBITS due to some edge case, truncate.
if len(SELECTED_FEATURES) > N_QUBITS:
    # This ensures uniqueness and prioritizes importance for features beyond the forced one
    final_selection_set = set(SELECTED_FEATURES[:1]) # Start with forced_feature
    for feat in importance_df['feature']:
        if feat not in final_selection_set:
            final_selection_set.add(feat)
        if len(final_selection_set) == N_QUBITS:
            break
    # Reorder based on original importance_df order for consistency
    SELECTED_FEATURES = [f for f in importance_df['feature'] if f in final_selection_set]
elif len(SELECTED_FEATURES) < N_QUBITS:
    # This case should ideally not happen if FEATURES_ALL is complete and N_QUBITS is reasonable
    print(f"Warning: Could not select {N_QUBITS} features. Only {len(SELECTED_FEATURES)} selected.")

assert len(SELECTED_FEATURES) == N_QUBITS, f"Expected {N_QUBITS} features, but got {len(SELECTED_FEATURES)}: {SELECTED_FEATURES}"
# --- End custom logic ---

importance_path = save_csv(importance_df, "feature_importances.csv")
selected_path = ARTIFACT_DIR / "selected_features.json"
write_json(selected_path, {
    "method": "ExtraTreesClassifier feature_importances_",
    "trained_only_on": "selector_pool",
    "n_qubits": N_QUBITS,
    "selected_features": SELECTED_FEATURES,
    "model": {
        "n_estimators": 500,
        "random_state": SEED_FEATURE_MODEL,
        "class_weight": "balanced",
    },
})

print("Variables seleccionadas / qubits:")
for q, feature in enumerate(SELECTED_FEATURES):
    print(f"q[{q}] <- {feature}")
display(importance_df)

## 5. Crear y bloquear un subconjunto cuántico nuevo

Se seleccionan 32 filas de cada clase desde el pool candidato cuántico. No se usa SMOTE ni ninguna muestra sintética.

In [ ]:
rng = np.random.default_rng(SEED_QUANTUM_SAMPLE)
qpool = df.iloc[quantum_candidate_idx].copy()

chosen = []
for label in [0, 1]:
    candidates = qpool.loc[qpool[TARGET] == label, "source_index"].to_numpy()
    chosen.extend(rng.choice(candidates, size=QUANTUM_ROWS // 2, replace=False).tolist())

chosen = np.array(chosen, dtype=int)
rng.shuffle(chosen)
quantum64 = df.set_index("source_index").loc[chosen].reset_index()
quantum64.insert(0, "quantum_row_id", np.arange(len(quantum64)))

assert len(quantum64) == QUANTUM_ROWS
assert quantum64[TARGET].value_counts().to_dict() == {0: 32, 1: 32}
assert set(quantum64["source_index"]).issubset(set(quantum_candidate_idx))
assert not set(quantum64["source_index"]) & set(holdout_idx)

q64_raw_path = save_csv(quantum64, "pytket_quantum64_raw.csv")
q64_selected_path = save_csv(
    quantum64[["quantum_row_id", "source_index", *SELECTED_FEATURES, TARGET]],
    "pytket_quantum64_selected.csv",
)

folds = np.full(QUANTUM_ROWS, -1, dtype=int)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED_FOLDS)
for fold, (_, val_pos) in enumerate(skf.split(np.zeros(QUANTUM_ROWS), quantum64[TARGET])):
    folds[val_pos] = fold

fold_df = quantum64[["quantum_row_id", "source_index", TARGET]].copy()
fold_df["fold"] = folds
fold_path = save_csv(fold_df, "pytket_quantum64_folds.csv")

assert (folds >= 0).all()
for fold in range(N_SPLITS):
    vc = fold_df.loc[fold_df["fold"] == fold, TARGET].value_counts().to_dict()
    assert vc == {0: 8, 1: 8}

display(fold_df.groupby(["fold", TARGET]).size().unstack(fill_value=0))

## 6. Crear el lock criptográfico

Los archivos críticos quedan identificados mediante SHA-256. Una modificación en el dataset, los splits, las variables, las 64 filas o los folds hará fallar la verificación.

In [ ]:
critical_paths = [
    SNAPSHOT_PATH,
    *paths_initial,
    importance_path,
    selected_path,
    q64_raw_path,
    q64_selected_path,
    fold_path,
]
lock_records = [file_record(p) for p in critical_paths]
LOCK_PATH = ARTIFACT_DIR / "DATA_LOCK.json"
write_json(LOCK_PATH, {
    "lock_version": 1,
    "dataset_source_path": str(DATASET_PATH),
    "dataset_sha256": raw_bytes_hash,
    "only_external_input": DATASET_PATH.name,
    "records": lock_records,
})
verify_records(lock_records)
print("Lock verificado:", LOCK_PATH)

## 7. Mapa de características personalizado en `pytket`

Cada atributo seleccionado controla un qubit. Primero se aplica Hadamard; luego cada capa contiene:

- una rotación local \(R_z(\alpha x_i)\);
- interacciones de dos qubits implementadas como `CX–Rz–CX`;
- una recarga no conmutativa mediante \(R_y(\alpha x_i/2)\).

`pytket` expresa los ángulos de rotación en **medias vueltas**: el parámetro `1.0` equivale a \(\pi\) radianes. Por eso los ángulos en radianes se dividen por \(\pi\) al construir el circuito.

In [ ]:
@dataclass(frozen=True)
class MapConfig:
    name: str
    reps: int
    topology: str
    alpha: float = 1.0

MAPS = [
    MapConfig("pytket_linear_r1", reps=1, topology="linear"),
    MapConfig("pytket_ring_r1", reps=1, topology="ring"),
    MapConfig("pytket_linear_r2", reps=2, topology="linear"),
]


def topology_edges(n_qubits: int, topology: str) -> list[tuple[int, int]]:
    if topology == "linear":
        return [(i, i + 1) for i in range(n_qubits - 1)]
    if topology == "ring":
        edges = [(i, i + 1) for i in range(n_qubits - 1)]
        if n_qubits > 2:
            edges.append((n_qubits - 1, 0))
        return edges
    raise ValueError(f"Topología desconocida: {topology}")


def build_pytket_feature_map(x_angles: np.ndarray, cfg: MapConfig) -> Circuit:
    x = np.asarray(x_angles, dtype=float)
    if x.ndim != 1 or len(x) != N_QUBITS:
        raise ValueError(f"Se esperaban {N_QUBITS} ángulos, recibido {x.shape}")

    circ = Circuit(N_QUBITS, name=cfg.name)
    for q in range(N_QUBITS):
        circ.H(q)

    for _ in range(cfg.reps):
        for q, angle in enumerate(x):
            circ.Rz(cfg.alpha * angle / np.pi, q)

        for i, j in topology_edges(N_QUBITS, cfg.topology):
            # Ángulo de interacción acotado: x_i*x_j/pi en [0, pi].
            pair_angle = cfg.alpha * x[i] * x[j] / np.pi
            circ.CX(i, j)
            circ.Rz(pair_angle / np.pi, j)
            circ.CX(i, j)

        for q, angle in enumerate(x):
            circ.Ry(0.5 * cfg.alpha * angle / np.pi, q)

    return circ


def statevectors_from_angles(X_angles: np.ndarray, cfg: MapConfig) -> np.ndarray:
    states = []
    for row in np.asarray(X_angles, dtype=float):
        state = np.asarray(build_pytket_feature_map(row, cfg).get_statevector(), dtype=complex)
        state = state / np.linalg.norm(state)
        states.append(state)
    return np.vstack(states)


def fidelity_kernel(A_states: np.ndarray, B_states: np.ndarray | None = None) -> np.ndarray:
    if B_states is None:
        B_states = A_states
    overlaps = A_states.conj() @ B_states.T
    K = np.abs(overlaps) ** 2
    return np.clip(K.real, 0.0, 1.0)

In [ ]:
# Diagrama del mapa base para un punto representativo.
selector_selected_imp = SimpleImputer(strategy="median").fit_transform(selector_df[SELECTED_FEATURES])
selector_angle_scaler = MinMaxScaler(feature_range=(0.0, np.pi), clip=True).fit(selector_selected_imp)
reference_raw = np.nanmedian(selector_df[SELECTED_FEATURES].to_numpy(dtype=float), axis=0).reshape(1, -1)
reference_imp = SimpleImputer(strategy="median").fit(selector_df[SELECTED_FEATURES]).transform(reference_raw)
reference_angles = selector_angle_scaler.transform(reference_imp)[0]

BASE_CFG = MAPS[0]
base_circuit = build_pytket_feature_map(reference_angles, BASE_CFG)
print(base_circuit)
print({
    "qubits": base_circuit.n_qubits,
    "depth": base_circuit.depth(),
    "two_qubit_depth": base_circuit.depth_2q(),
    "cx_count": base_circuit.n_gates_of_type(OpType.CX),
    "two_qubit_gates": base_circuit.n_2qb_gates(),
})

try:
    render_circuit_jupyter(base_circuit)
except Exception as exc:
    print("El render interactivo no estuvo disponible; el circuito textual permanece arriba:", exc)

(ARTIFACT_DIR / "pytket_baseline_circuit.qasm").write_text(
    circuit_to_qasm_str(base_circuit), encoding="utf-8"
)
try:
    html = render_circuit_as_html(base_circuit)
    if html:
        (ARTIFACT_DIR / "pytket_baseline_circuit.html").write_text(html, encoding="utf-8")
except Exception as exc:
    print("No se pudo guardar HTML del circuito:", exc)

## 8. Validación cruzada limpia

En cada fold externo:

1. la mediana se aprende con las 48 filas de entrenamiento;
2. el escalado angular a \([0,\pi]\) se aprende con esas mismas 48 filas;
3. se calculan \(K_{train}\) y \(K_{validation}\);
4. `C` se selecciona mediante tres folds internos sobre `K_train`;
5. se evalúan las 16 filas externas.

In [ ]:
def tune_c_precomputed(K: np.ndarray, y: np.ndarray, seed: int) -> tuple[float, pd.DataFrame]:
    rows = []
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    for C in [0.1, 1.0, 10.0]:
        scores = []
        for tr, va in inner.split(K, y):
            model = SVC(kernel="precomputed", C=C, class_weight="balanced")
            model.fit(K[np.ix_(tr, tr)], y[tr])
            pred = model.predict(K[np.ix_(va, tr)])
            scores.append(f1_score(y[va], pred, zero_division=0))
        rows.append({"C": C, "inner_f1_mean": np.mean(scores), "inner_f1_std": np.std(scores)})
    table = pd.DataFrame(rows).sort_values(["inner_f1_mean", "C"], ascending=[False, True])
    return float(table.iloc[0]["C"]), table


X64_raw = quantum64[SELECTED_FEATURES].to_numpy(dtype=float)
y64 = quantum64[TARGET].to_numpy(dtype=int)

fold_rows = []
inner_rows = []
kernels_by_map_fold = {}

for cfg in MAPS:
    print("Evaluando", cfg.name)
    for fold in range(N_SPLITS):
        va = np.flatnonzero(folds == fold)
        tr = np.flatnonzero(folds != fold)

        imputer = SimpleImputer(strategy="median")
        Xtr_imp = imputer.fit_transform(X64_raw[tr])
        Xva_imp = imputer.transform(X64_raw[va])

        scaler = MinMaxScaler(feature_range=(0.0, np.pi), clip=True)
        Xtr_angles = scaler.fit_transform(Xtr_imp)
        Xva_angles = scaler.transform(Xva_imp)

        t0 = time.perf_counter()
        Str = statevectors_from_angles(Xtr_angles, cfg)
        Sva = statevectors_from_angles(Xva_angles, cfg)
        state_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        Ktr = fidelity_kernel(Str)
        Kva = fidelity_kernel(Sva, Str)
        kernel_time = time.perf_counter() - t0

        assert np.allclose(Ktr, Ktr.T, atol=1e-10)
        assert np.allclose(np.diag(Ktr), 1.0, atol=1e-10)

        best_c, c_table = tune_c_precomputed(Ktr, y64[tr], seed=SEED_FOLDS + fold)
        c_table.insert(0, "map", cfg.name)
        c_table.insert(1, "outer_fold", fold)
        inner_rows.append(c_table)

        model = SVC(kernel="precomputed", C=best_c, class_weight="balanced")
        t0 = time.perf_counter()
        model.fit(Ktr, y64[tr])
        pred = model.predict(Kva)
        svc_time = time.perf_counter() - t0

        row = {
            "map": cfg.name,
            "fold": fold,
            "best_C": best_c,
            "train_rows": len(tr),
            "validation_rows": len(va),
            "statevector_seconds": state_time,
            "kernel_seconds": kernel_time,
            "svc_seconds": svc_time,
            **classification_metrics(y64[va], pred),
        }
        fold_rows.append(row)
        kernels_by_map_fold[(cfg.name, fold)] = {"K_train": Ktr, "K_validation": Kva, "train_pos": tr, "val_pos": va}

fold_metrics = pd.DataFrame(fold_rows)
inner_c_results = pd.concat(inner_rows, ignore_index=True)
summary = (
    fold_metrics.groupby("map")
    .agg(
        accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"), balanced_accuracy_std=("balanced_accuracy", "std"),
        f1_mean=("f1", "mean"), f1_std=("f1", "std"),
        mcc_mean=("mcc", "mean"), mcc_std=("mcc", "std"),
        total_seconds=("statevector_seconds", "sum"),
    )
    .sort_values(["f1_mean", "balanced_accuracy_mean"], ascending=False)
    .reset_index()
)

save_csv(fold_metrics, "pytket_fold_metrics.csv")
save_csv(inner_c_results, "pytket_inner_c_search.csv")
save_csv(summary, "pytket_cv_summary.csv")
display(summary)

## 9. Geometría global descriptiva

Para visualización, el preprocesamiento se ajusta únicamente en el pool selector independiente y se aplica a las 64 filas. Estas matrices no se usan para calcular las métricas externas anteriores.

In [ ]:
def centered_kernel_alignment(K: np.ndarray, y: np.ndarray) -> float:
    ypm = np.where(y == 1, 1.0, -1.0)
    Y = np.outer(ypm, ypm)
    H = np.eye(len(K)) - np.ones_like(K) / len(K)
    Kc = H @ K @ H
    Yc = H @ Y @ H
    denom = np.linalg.norm(Kc, "fro") * np.linalg.norm(Yc, "fro")
    return float(np.sum(Kc * Yc) / denom) if denom else np.nan


def kernel_geometry(K: np.ndarray, y: np.ndarray) -> dict:
    same = y[:, None] == y[None, :]
    offdiag = ~np.eye(len(y), dtype=bool)
    intra = K[same & offdiag]
    inter = K[(~same) & offdiag]
    eigvals = np.linalg.eigvalsh((K + K.T) / 2)
    eigvals = np.clip(eigvals, 0.0, None)
    p = eigvals / eigvals.sum() if eigvals.sum() else eigvals
    effective_rank = float(np.exp(-(p[p > 0] * np.log(p[p > 0])).sum())) if np.any(p > 0) else 0.0
    return {
        "alignment": centered_kernel_alignment(K, y),
        "intra_mean": float(intra.mean()),
        "inter_mean": float(inter.mean()),
        "intra_minus_inter": float(intra.mean() - inter.mean()),
        "effective_rank": effective_rank,
        "min_eigenvalue": float(eigvals.min()),
        "max_eigenvalue": float(eigvals.max()),
    }

# Preprocesador descriptivo independiente de las 64 filas.
global_imputer = SimpleImputer(strategy="median").fit(selector_df[SELECTED_FEATURES])
selector_imp = global_imputer.transform(selector_df[SELECTED_FEATURES])
global_scaler = MinMaxScaler(feature_range=(0.0, np.pi), clip=True).fit(selector_imp)
X64_global_angles = global_scaler.transform(global_imputer.transform(quantum64[SELECTED_FEATURES]))

global_kernels = {}
geometry_rows = []
resource_rows = []

for cfg in MAPS:
    t0 = time.perf_counter()
    states = statevectors_from_angles(X64_global_angles, cfg)
    K = fidelity_kernel(states)
    elapsed = time.perf_counter() - t0
    global_kernels[cfg.name] = K

    geometry_rows.append({"map": cfg.name, "kernel_seconds": elapsed, **kernel_geometry(K, y64)})

    circ = build_pytket_feature_map(reference_angles, cfg)
    resource_rows.append({
        **asdict(cfg),
        "n_qubits": circ.n_qubits,
        "depth": circ.depth(),
        "two_qubit_depth": circ.depth_2q(),
        "cx_count": circ.n_gates_of_type(OpType.CX),
        "two_qubit_gate_count": circ.n_2qb_gates(),
        "total_gate_count": len(circ.get_commands()),
    })
    (ARTIFACT_DIR / f"circuit_{cfg.name}.qasm").write_text(circuit_to_qasm_str(circ), encoding="utf-8")

    order = np.argsort(y64, kind="stable")
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(K[np.ix_(order, order)], vmin=0, vmax=1, aspect="auto")
    ax.set_title(f"Kernel de fidelidad — {cfg.name}\nordenado por clase")
    ax.set_xlabel("Observación")
    ax.set_ylabel("Observación")
    fig.colorbar(im, ax=ax, label="K(i,j)")
    fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / f"heatmap_{cfg.name}.png", dpi=180)
    plt.show()

geometry_df = pd.DataFrame(geometry_rows).sort_values("alignment", ascending=False)
resources_df = pd.DataFrame(resource_rows)
save_csv(geometry_df, "pytket_kernel_geometry.csv")
save_csv(resources_df, "pytket_circuit_resources.csv")
np.savez_compressed(ARTIFACT_DIR / "pytket_global_kernels.npz", **global_kernels)

display(geometry_df)
display(resources_df)

## 10. Sensibilidad compacta a shots

Esta sección no ejecuta hardware: toma la fidelidad exacta del mapa base como probabilidad y simula una estimación binomial con un número finito de shots. Sirve para medir cuánto se perturba la matriz por error de muestreo.

In [ ]:
base_K = global_kernels[BASE_CFG.name]
shot_rng = np.random.default_rng(SEED_SHOTS)
shot_rows = []

for shots in [128, 512, 2048]:
    for repeat in range(20):
        Kshot = np.eye(QUANTUM_ROWS)
        for i in range(QUANTUM_ROWS):
            for j in range(i + 1, QUANTUM_ROWS):
                estimate = shot_rng.binomial(shots, base_K[i, j]) / shots
                Kshot[i, j] = Kshot[j, i] = estimate
        eig_min = np.linalg.eigvalsh((Kshot + Kshot.T) / 2).min()
        shot_rows.append({
            "map": BASE_CFG.name,
            "shots": shots,
            "repeat": repeat,
            "mae_vs_exact": float(np.mean(np.abs(Kshot - base_K))),
            "max_error_vs_exact": float(np.max(np.abs(Kshot - base_K))),
            "min_eigenvalue": float(eig_min),
            "alignment": centered_kernel_alignment(Kshot, y64),
        })

shot_df = pd.DataFrame(shot_rows)
shot_summary = shot_df.groupby("shots").agg(
    mae_mean=("mae_vs_exact", "mean"),
    mae_std=("mae_vs_exact", "std"),
    min_eigenvalue_mean=("min_eigenvalue", "mean"),
    alignment_mean=("alignment", "mean"),
).reset_index()
save_csv(shot_df, "pytket_shot_sensitivity.csv")
save_csv(shot_summary, "pytket_shot_sensitivity_summary.csv")
display(shot_summary)

## 11. Manifiesto final y verificación

El manifiesto documenta las semillas, particiones, variables, mapas y versiones. Los hashes finales permiten comprobar que una ejecución posterior utiliza exactamente los mismos datos bloqueados.

In [ ]:
# Guardar hashes de los kernels por fold sin depender de pickle.
fold_kernel_records = []
for (map_name, fold), payload in kernels_by_map_fold.items():
    path = ARTIFACT_DIR / f"kernel_{map_name}_fold{fold}.npz"
    np.savez_compressed(path, **payload)
    fold_kernel_records.append(file_record(path))

all_artifacts = sorted([p for p in ARTIFACT_DIR.iterdir() if p.is_file() and p.name != "FINAL_MANIFEST.json"])

critical_lock_content = None
if LOCK_PATH.exists():
    try:
        critical_lock_content = json.loads(LOCK_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        print(f"Warning: {LOCK_PATH} found but its content is not valid JSON. Setting critical_lock to null.")
else:
    print(f"Warning: {LOCK_PATH} not found. Setting critical_lock to null. This may indicate an issue with previous steps or manual deletion.")

manifest = {
    "project": "Water Potability quantum kernel with pytket",
    "methodology": "from scratch; only water_potability.csv is read",
    "dataset": {
        "source": str(DATASET_PATH),
        "sha256": raw_bytes_hash,
        "rows": len(df),
        "features_original": FEATURES_ALL,
    },
    "seeds": {
        "split": SEED_SPLIT,
        "selector_split": SEED_SELECTOR_SPLIT,
        "feature_model": SEED_FEATURE_MODEL,
        "quantum_sample": SEED_QUANTUM_SAMPLE,
        "folds": SEED_FOLDS,
        "shots": SEED_SHOTS,
    },
    "split_sizes": {
        "selector_pool": len(selector_idx),
        "quantum_candidate_pool": len(quantum_candidate_idx),
        "locked_holdout": len(holdout_idx),
        "quantum64": len(quantum64),
    },
    "selected_features": SELECTED_FEATURES,
    "n_qubits": N_QUBITS,
    "maps": [asdict(cfg) for cfg in MAPS],
    "svc": {"kernel": "precomputed", "C_grid": [0.1, 1.0, 10.0], "class_weight": "balanced"},
    "pytket_version": getattr(sys.modules.get("pytket"), "__version__", "unknown"),
    "critical_lock": critical_lock_content,
    "fold_kernel_records": fold_kernel_records,
    "artifacts": [file_record(p) for p in all_artifacts],
}
FINAL_MANIFEST = ARTIFACT_DIR / "FINAL_MANIFEST.json"
write_json(FINAL_MANIFEST, manifest)

# Controles finales.
verify_records(lock_records)
assert len(SELECTED_FEATURES) == N_QUBITS
assert quantum64[TARGET].value_counts().to_dict() == {0: 32, 1: 32}
assert not set(quantum64["source_index"]) & set(holdout_idx)
assert not any(name.startswith("qiskit") for name in sys.modules)

checks = pd.DataFrame([
    {"check": "only_dataset_as_external_input", "passed": True},
    {"check": "dataset_snapshot_hash_matches", "passed": sha256_file(SNAPSHOT_PATH) == raw_bytes_hash},
    {"check": "all_primary_splits_disjoint", "passed": True},
    {"check": "feature_selection_independent_of_quantum64", "passed": True},
    {"check": "four_features_four_qubits", "passed": len(SELECTED_FEATURES) == N_QUBITS},
    {"check": "quantum64_balanced", "passed": quantum64[TARGET].value_counts().to_dict() == {0: 32, 1: 32}},
    {"check": "quantum64_has_no_holdout_rows", "passed": not bool(set(quantum64["source_index"]) & set(holdout_idx))},
    {"check": "no_qiskit_imported", "passed": not any(name.startswith("qiskit") for name in sys.modules)},
])
save_csv(checks, "pytket_reproducibility_checks.csv")
assert checks["passed"].all()

display(checks)
print("Ejecución completa. Artefactos:", ARTIFACT_DIR)

## Interpretación esperada

- Una mayor expresividad no garantiza mejor F1: dos capas pueden concentrar el kernel o volverlo demasiado cercano a la identidad.
- El mapa lineal usa menos compuertas de dos qubits que el anillo.
- La matriz exacta debe ser simétrica, con diagonal uno y autovalores no negativos salvo errores numéricos.
- Las estimaciones por shots pueden perder semidefinitud positiva; esto se observa mediante el autovalor mínimo.
- El holdout bloqueado no se evalúa aquí. Su propósito es permitir una evaluación final posterior, después de congelar el diseño del mapa.